# Modelo para la creacion, mediante PDFs

### Primero tomamos un pdf y extraemos el texto

In [1]:
import pdfplumber

def extraer_texto_pdf(ruta_pdf):
    texto = ""
    
    with pdfplumber.open(ruta_pdf) as pdf:
        for pagina in pdf.pages:
            contenido = pagina.extract_text()
            if contenido:
                texto += contenido + "\n"
    
    return texto


# Ejemplo de uso
ruta = "transformers_teoria.pdf"
texto = extraer_texto_pdf(ruta)

print(texto)

Redes Neuronales Transformers: Explicación
Teórica
Los Transformers son una arquitectura de redes neuronales introducida en 2017 en el artículo
“Attention is All You Need”.
Se utilizan principalmente en tareas de procesamiento de lenguaje natural, aunque también se
aplican en visión y otras áreas.
A diferencia de modelos anteriores como las redes recurrentes (RNN) o LSTM, los Transformers
no procesan la información de manera secuencial.
En su lugar, utilizan un mecanismo llamado “atención” que les permite considerar todas las partes
de la entrada al mismo tiempo.
El concepto clave de los Transformers es la atención, específicamente la “self-attention” o
autoatención.
Este mecanismo permite que cada palabra en una secuencia evalúe la importancia de las demás
palabras dentro del mismo contexto.
Gracias a esto, el modelo puede capturar relaciones a largo plazo sin depender de pasos
secuenciales.
La arquitectura básica de un Transformer se compone de dos partes principales: el encoder y el

### Separa la letra de la cancion de fondo

In [12]:
import subprocess
import os

def separar_audio(ruta_audio):
    try:
        # Ejecuta Demucs desde Python
        subprocess.run(
            ["demucs", ruta_audio],
            check=True
        )

        print("Separación completada.")

        # Ruta donde Demucs guarda los resultados
        nombre = os.path.splitext(os.path.basename(ruta_audio))[0]
        salida = f"separated/htdemucs/{nombre}"

        print("Archivos generados en:", salida)
        print("Vocales:", os.path.join(salida, "vocals.wav"))
        print("Música:", os.path.join(salida, "no_vocals.wav"))

        return salida

    except subprocess.CalledProcessError as e:
        print("Error al ejecutar Demucs:", e)

# --- USO ---
ruta = "example.mp3"
separar_audio(ruta)

Error al ejecutar Demucs: Command '['demucs', 'example.mp3']' returned non-zero exit status 1.


#### BPM

In [2]:
import librosa

y, sr = librosa.load("example.mp3")

# Detectar tempo (BPM)
tempo, beats = librosa.beat.beat_track(y=y, sr=sr)

print("BPM:", tempo)

C:\Users\luigu\anaconda3\envs\MusicaP\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BPM: [99.38401442]


#### Duracion

### Crear la letra con LLM

In [10]:
import boto3
from botocore.exceptions import ClientError

def llamar_qwen(prompt):
    # 1. Configurar el cliente de Bedrock Runtime
    # Asegúrate de que la región sea donde Qwen esté disponible (ej. us-east-1 o us-west-2)
    client = boto3.client("bedrock-runtime", region_name="us-east-1")

    # 2. Definir el ID del modelo Qwen
    # Ejemplo para Qwen 2.5 72B Instruct (verifica la versión exacta en tu catálogo)
    model_id = "qwen.qwen3-next-80b-a3b"

    # 3. Preparar el mensaje
    messages = [
        {
            "role": "user",
            "content": [{"text": prompt}]
        }
    ]

    try:
        # 4. Realizar la llamada usando la Converse API
        response = client.converse(
            modelId=model_id,
            messages=messages,
            inferenceConfig={
                "maxTokens": 1000,
                "temperature": 0.7,
                "topP": 0.9
            }
        )

        # 5. Extraer y retornar el texto
        return response['output']['message']['content'][0]['text']

    except ClientError as e:
        return f"Error de cliente: {e}"
    except Exception as e:
        return f"Error inesperado: {e}"
prompt = """Te voy a dar un texto puede ser de lo que sea, tu debes tomar la informacion mas importante y convertirla en una letra de un cancion.
                    Reglas:
                    - No incluyas explicaciones
                    - No incluyas frases como "aquí tienes"
                    - No agregues comentarios
                    - No uses markdown
                    - Solo texto plano de la letra"""
# --- Ejemplo de uso ---
if __name__ == "__main__":
    pregunta =  prompt + texto
    resultado = llamar_qwen(pregunta)
    print("Respuesta de Qwen:\n", resultado)

Respuesta de Qwen:
 Attention is all you need, no more chains of time  
Every word sees every word, in a single glance  
No left to right, no step by step  
Just layers of focus, breaking the grip  

Multi-head eyes, each one sees different ties  
Connections born where meaning lies  
Position encoded, the order stays  
Though sequence’s gone, the flow still plays  

Encoder builds what the input says  
Decoder dreams the answer in its ways  
One side learns the world, the other speaks  
Through attention’s power, the truth takes shape  

Residual paths, the gradients flow  
Deep networks rise where once they’d slow  
No more waiting, no more delay  
Parallel minds, they seize the day  

From text to vision, from tongue to sight  
They changed the game with pure attention’s light  
No RNN’s chain, no LSTM’s hold  
Just attention—bold, and infinite, and bold


El BPM determina el “mood”:

| BPM        | Descripción                          |
|------------|--------------------------------------|
| 60–80 BPM  | → lento (baladas, música triste)     |
| 90–110 BPM | → relajado (hip hop, pop tranquilo)  |
| 120–130 BPM| → bailable (electrónica, pop)        |
| 140+ BPM   | → muy energético (trap, techno)      |

In [5]:
from TTS.api import TTS

# cargar modelo XTTS
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2")

# generar audio
tts.tts_to_file(
    text= texto,
    file_path="salida.wav"
)

C:\Users\luigu\anaconda3\envs\UltimateMusic\Lib\site-packages\jieba\_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


 > You must confirm the following:
 | > "I have purchased a commercial license from Coqui: licensing@coqui.ai"
 | > "Otherwise, I agree to the terms of the non-commercial CPML: https://coqui.ai/cpml" - [y/n]


 | | >  Y


 > Downloading model to C:\Users\luigu\AppData\Local\tts\tts_models--multilingual--multi-dataset--xtts_v2


100%|█████████▉| 1.87G/1.87G [02:10<00:00, 15.2MiB/s]
100%|██████████| 1.87G/1.87G [02:10<00:00, 14.3MiB/s]
4.37kiB [00:00, 8.51kiB/s]

361kiB [00:00, 626kiB/s].0 [00:00<?, ?iB/s]
100%|██████████| 32.0/32.0 [00:00<00:00, 57.8iB/s]
 99%|█████████▊| 7.65M/7.75M [00:00<00:00, 17.5MiB/s]C:\Users\luigu\anaconda3\envs\UltimateMusic\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 > Model's license - CPML
 > Check https://coqui.ai/cpml.txt for more info.


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

100%|██████████| 7.75M/7.75M [00:17<00:00, 17.5MiB/s]